In [1]:
import pandas as pd 
from sqlalchemy import create_engine

In [2]:
USERNAME = "postgres"
PASSWORD = "admin"
HOST = "localhost"
PORT = "5432"
DATABASE = "ecommerce_database"


In [3]:
engine = create_engine(
    f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
)


In [4]:
conn = engine.connect()

#### executing analytical query 

1. Monthly GMV trend — total payment value by order month with MoM growth %

In [5]:
query="""
set search_path to olist;

WITH MONTHLY_GMV AS (

SELECT 
	DATE_TRUNC('month',O.ORDER_PURCHASE_TIMESTAMP) AS ORDER_MONTH,
	ROUND(SUM(OP.PAYMENT_VALUE),2) AS TOTAL_MONTHLY_GMV
FROM ORDER_PAYMENTS AS OP
JOIN ORDERS AS O ON OP.ORDER_ID=O.ORDER_ID 
WHERE O.ORDER_STATUS='delivered'
GROUP BY ORDER_MONTH), 

GMV_GROWTH AS (
SELECT 
	ORDER_MONTH,
	TOTAL_MONTHLY_GMV,
	LAG(TOTAL_MONTHLY_GMV) OVER(ORDER BY ORDER_MONTH) AS PREVIOUS_MONTH_GMV
FROM MONTHLY_GMV
)

SELECT 
	ORDER_MONTH,
	ROUND(
		((TOTAL_MONTHLY_GMV-PREVIOUS_MONTH_GMV) /TOTAL_MONTHLY_GMV)*100 
		,2) AS GMV_PERCENTAGE_GROWTH
FROM GMV_GROWTH 
ORDER BY ORDER_MONTH
OFFSET 2

"""

In [6]:
df = pd.read_sql_query(query, conn)

df

,order_month,gmv_percentage_growth
0,2017-01-01,99.98
1,2017-02-01,52.99
2,2017-03-01,34.53
3,2017-04-01,-5.99
4,2017-05-01,31.06
5,2017-06-01,-15.67
6,2017-07-01,13.45
7,2017-08-01,12.32
8,2017-09-01,7.87
9,2017-10-01,6.65


### 2. Average Order Value (AOV) per product category and per state

In [7]:
query="""
select c.customer_state,eng.product_category_name_english,AVG(pay.payment_value) as average_order_value
from orders as o
join customers as c on c.customer_id=o.customer_id
join order_payments as pay  on pay.order_id=o.order_id
join order_items as itm on o.order_id =itm.order_id
join products as p on p.product_id=itm.product_id
join product_category_name_translation as eng on eng.product_category_name=p.product_category_name
where o.order_status='delivered'
group by c.customer_state,eng.product_category_name
order by AVG(pay.payment_value) desc
"""

In [8]:
df = pd.read_sql_query(query, conn)
print(df.head(4))

  customer_state product_category_name_english  average_order_value
0             RJ               fixed_telephony             3388.665
1             MT        signaling_and_security             3242.840
2             ES               fixed_telephony             3024.854
3             PB        signaling_and_security             2844.960


Q3

In [9]:
query = """
select eng.product_category_name_english as product_category , sum(pay.payment_value) as revenue
from orders as o 
join order_payments as pay  on pay.order_id=o.order_id
join order_items as itm on o.order_id =itm.order_id
join products as p on p.product_id=itm.product_id
join product_category_name_translation as eng on eng.product_category_name=p.product_category_name
where o.order_status='delivered'
group by product_category
order by revenue desc
limit 10 
"""

In [10]:
df=pd.read_sql(query, conn)
df

,product_category,revenue
0,bed_bath_table,1692714.28
1,health_beauty,1620684.04
2,computers_accessories,1549372.59
3,furniture_decor,1394466.93
4,watches_gifts,1387362.45
5,sports_leisure,1349446.93
6,housewares,1069787.97
7,auto,833745.67
8,garden_tools,810614.93
9,cool_stuff,744649.32


Q9--

In [11]:
query="""
select c.customer_state,eng.product_category_name_english as product_category ,AVG(rev.review_score) as average_review_score
from orders as o
join customers as c on c.customer_id=o.customer_id
join order_reviews as rev  on rev.order_id=o.order_id
join order_items as itm on o.order_id =itm.order_id
join products as p on p.product_id=itm.product_id
join product_category_name_translation as eng on eng.product_category_name=p.product_category_name
where o.order_status='delivered'
group by c.customer_state,product_category
"""

In [12]:
df=pd.read_sql(query, conn)
df

,customer_state,product_category,average_review_score
0,AC,auto,3.500000
1,AC,baby,5.000000
2,AC,bed_bath_table,2.500000
3,AC,books_general_interest,4.000000
4,AC,christmas_supplies,4.000000
...,...,...,...
1341,TO,sports_leisure,3.920000
1342,TO,stationery,5.000000
1343,TO,telephony,3.809524
1344,TO,toys,4.153846


Q4

In [13]:
query ="""
select eng.product_category_name_english as product_category,
	AVG(
		ROUND(
			(ot.freight_value/ot.price)*100
		,2)
	) as Average_percentage_freight_value
from order_items as ot
join products as p on ot.product_id=p.product_id
join product_category_name_translation as eng on eng.product_category_name=p.product_category_name
group by product_category
order by Average_percentage_freight_value desc

"""

In [14]:
df = pd.read_sql_query(query, conn)
df


,product_category,average_percentage_freight_value
0,home_comfort_2,93.385000
1,dvds_blu_ray,83.303594
2,electronics,68.351601
3,christmas_supplies,67.565752
4,fashion_underwear_beach,56.550076
...,...,...
66,la_cuisine,18.596429
67,watches_gifts,17.058221
68,small_appliances_home_oven_and_coffee,16.434868
69,security_and_services,14.755000


--Q5

In [15]:
query="""
select 
    payment_type,
	CASE
		WHEN payment_installments=1 THEN 'full payment'
		ELSE 'on installment'
 	END AS Payment_installment,
	
 	sum(payment_value) as revenve 
from order_payments as op
group by payment_type,
	CASE
	WHEN payment_installments=1 THEN 'full payment'
	ELSE 'on installment'
	END
having sum(payment_value)>0

"""

In [16]:
df=pd.read_sql(query, conn)

--Q6

In [17]:
query="""
select 
	EXTRACT(HOURS FROM o.ORDER_PURCHASE_TIMESTAMP) as hour_of_the_day,
	TO_CHAR(o.ORDER_PURCHASE_TIMESTAMP,'Day') as Day_of_week, 
	count(o.order_id) as number_of_orders
from orders as o 
where o.order_status='delivered'
group by EXTRACT(HOURS FROM o.ORDER_PURCHASE_TIMESTAMP),TO_CHAR(o.ORDER_PURCHASE_TIMESTAMP,'Day')
order by number_of_orders desc
"""

In [18]:
pd.read_sql_query(query, conn)

,hour_of_the_day,day_of_week,number_of_orders
0,14.0,Tuesday,1095
1,21.0,Monday,1083
2,14.0,Monday,1063
3,16.0,Monday,1060
4,16.0,Tuesday,1059
...,...,...,...
163,5.0,Wednesday,24
164,5.0,Tuesday,24
165,5.0,Saturday,23
166,5.0,Monday,22


--Q8

In [19]:
query="""
select customer_id, count(order_id) as no_of_orders
from orders as o 
where order_status='delivered'
group by customer_id"""

In [20]:
df = pd.read_sql_query(query, conn)
df.tail(4)

,customer_id,no_of_orders
96474,fffecc9f79fd8c764f843e9951b11341,1
96475,fffeda5b6d849fbd39689bb92087f431,1
96476,ffff42319e9b2d713724ae527742af25,1
96477,ffffa3172527f765de70084a7e53aae8,1


Does late delivery explain low scores? Category avg score split by on-time vs late

In [21]:
query="""
with delivery_delay as (
select o.order_id,
	CASE 
	WHEN o.order_delivered_customer_date<=o.order_estimated_delivery_date THEN 'on_time' 
	ELSE 'delayed'
	END as  delivery_delay_status
from orders as o
where o.order_delivered_customer_date iS NOT NULL AND o.order_estimated_delivery_date is NOT NULL
)
,review_score as(
	select 
		eng.product_category_name_english as category,
		dd.delivery_delay_status ,
		COUNT(ore.review_id) as review_count,
		AVG(ore.review_score) as avg_review_score 
	
	from order_reviews as ore
	NATURAL JOIN delivery_delay as dd
	join order_items as oi on oi.order_id=ore.order_id
	join products as p on  oi.product_id=p.product_id
	join product_category_name_translation as eng on eng.product_category_name=p.product_category_name
	
	group by eng.product_category_name_english,dd.delivery_delay_status
	HAVING    COUNT(ore.review_id) >= 30
	)

select category , delivery_delay_status , review_count , ROUND(avg_review_score,2) as avg_review,
		ROUND(AVG(
		CASE 
		WHEN delivery_delay_status ='on_time'
		THEN avg_review_score
		END ) OVER(PARTITION BY category)
		- AVG(
		CASE 
		WHEN delivery_delay_status ='delayed' 
		THEN avg_review_score 
		END ) OVER(PARTITION BY category),2) as score_difference
		
	
from review_score

"""

In [22]:
df=pd.read_sql(query, conn)
df

,category,delivery_delay_status,review_count,avg_review,score_difference
0,agro_industry_and_commerce,on_time,197,4.16,NaN
1,air_conditioning,on_time,274,4.13,NaN
2,art,on_time,180,4.17,NaN
3,audio,delayed,45,1.96,2.14
4,audio,on_time,314,4.10,2.14
...,...,...,...,...,...
91,telephony,on_time,4043,4.12,1.50
92,toys,delayed,291,2.39,1.96
93,toys,on_time,3716,4.35,1.96
94,watches_gifts,delayed,474,2.40,1.82


Q25

In [23]:
query="""
WITH seller_category_sales AS (

    SELECT
        oi.seller_id,

        eng.product_category_name_english AS category,

        SUM(oi.price) AS category_revenue

    FROM order_items oi

    JOIN products p
        ON oi.product_id = p.product_id

    JOIN product_category_name_translation eng
        ON eng.product_category_name = p.product_category_name

    GROUP BY
        oi.seller_id,
        eng.product_category_name_english
),

seller_total_sales AS (

    SELECT
        seller_id,
        SUM(category_revenue) AS total_revenue

    FROM seller_category_sales

    GROUP BY seller_id
),

specialization AS (

    SELECT
        scs.seller_id,
        scs.category,
        scs.category_revenue,
        sts.total_revenue,

        ROUND(
            (scs.category_revenue / sts.total_revenue) * 100,
            2
        ) AS category_contribution_pct,

        RANK() OVER(
            PARTITION BY scs.seller_id
            ORDER BY scs.category_revenue DESC
        ) AS category_rank

    FROM seller_category_sales scs

    JOIN seller_total_sales sts
        ON scs.seller_id = sts.seller_id
)

SELECT
    seller_id,
    category AS dominant_category,
    category_revenue,
    total_revenue,
    category_contribution_pct

FROM specialization

WHERE category_rank = 1
  AND category_contribution_pct >= 70

ORDER BY category_contribution_pct DESC;"""

In [24]:
df=pd.read_sql(query, conn)
df

,seller_id,dominant_category,category_revenue,total_revenue,category_contribution_pct
0,0015a82c2db000af6aaaf3ae2ecb0532,small_appliances,2685.00,2685.00,100.00
1,966cb4760537b1404caedd472cc610a5,watches_gifts,51286.05,51286.05,100.00
2,968ee78631915a63fef426d6733d7422,cool_stuff,450.00,450.00,100.00
3,96f7c797de9ca20efbe14545bed63eec,perfumery,138.60,138.60,100.00
4,972d0f9cf61b499a4812cf0bfa3ad3c4,bed_bath_table,8089.29,8089.29,100.00
...,...,...,...,...,...
2430,7b98de631987e26dd6d803490c43a13c,construction_tools_construction,447.40,634.20,70.55
2431,ed859002ad59dbf8cf3602696a6c3000,air_conditioning,1706.90,2421.70,70.48
2432,c847e075301870dd144a116762eaff9a,auto,10799.10,15348.70,70.36
2433,d66c11a9572221d92fbb8c4897db5f9b,electronics,546.38,776.56,70.36


In [25]:
query="""
SELECT ENG.PRODUCT_CATEGORY_NAME_ENGLISH AS PRODUCT_CATEGORY_NAME,
	DATE_TRUNC('month',O.ORDER_PURCHASE_TIMESTAMP) AS ORDER_MONTH,
	ROUND(
	COUNT(CASE WHEN O.ORDER_STATUS!='delivered' THEN O.ORDER_ID END)/
	COUNT(O.ORDER_ID),2) * 100 AS ORDER_CANCALATION_RATE
FROM ORDERS AS O 
JOIN ORDER_ITEMS AS OI ON O.ORDER_ID=OI.ORDER_ID 
JOIN PRODUCTS AS P ON OI.PRODUCT_ID=P.PRODUCT_ID
JOIN PRODUCT_CATEGORY_NAME_TRANSLATION AS ENG ON ENG.PRODUCT_CATEGORY_NAME=P.PRODUCT_CATEGORY_NAME

GROUP BY ENG.PRODUCT_CATEGORY_NAME_ENGLISH ,DATE_TRUNC('month',O.ORDER_PURCHASE_TIMESTAMP)
ORDER BY ORDER_CANCALATION_RATE DESC
"""

In [26]:
df=pd.read_sql(query, conn)
df

,product_category_name,order_month,order_cancalation_rate
0,furniture_decor,2016-09-01,100.0
1,fashion_childrens_clothes,2017-08-01,100.0
2,books_general_interest,2016-10-01,100.0
3,construction_tools_safety,2017-03-01,100.0
4,books_imported,2017-02-01,100.0
...,...,...,...
1248,agro_industry_and_commerce,2017-07-01,0.0
1249,agro_industry_and_commerce,2017-08-01,0.0
1250,agro_industry_and_commerce,2017-09-01,0.0
1251,agro_industry_and_commerce,2017-10-01,0.0
